# acf_pacf_analysis.py
Generates ACF and PACF plots for selected ZIP codes to guide SARIMAX order selection.

In [ ]:
"""
ACF and PACF plots for six representative ZIPs to inform SARIMAX order selection.
Input:  outputs/sarima_data.csv
Output: outputs/acf_pacf/acf_pacf_{zip}.png
"""
from pathlib import Path
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
INPUT_FILE = PROJECT_ROOT / "outputs" / "sarima_data.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "acf_pacf"
TARGET_ZIPS = ["02126", "02116", "02150", "02139", "01840", "02481"]
ZIP_LABELS  = {
    "02126": "Mattapan", "02116": "Back Bay",
    "02150": "Chelsea",  "02139": "Cambridge",
    "01840": "Lawrence", "02481": "Wellesley",
}
df = pd.read_csv(INPUT_FILE, dtype={"zip_code": str})
df["date"] = pd.to_datetime(df["date"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for zip_code in TARGET_ZIPS:
    series = (
        df[df["zip_code"] == zip_code]
        .sort_values("date")
        .set_index("date")["zhvi"]
        .diff()
        .dropna()
    )
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
    fig.suptitle(f"{zip_code} — {ZIP_LABELS[zip_code]} | Differenced ZHVI", fontweight="bold")
    plot_acf(series,  lags=36, ax=ax1)
    plot_pacf(series, lags=36, ax=ax2, method="ywm")
    ax1.set_title("ACF  →  suggests q and Q")
    ax2.set_title("PACF →  suggests p and P")
    for ax in (ax1, ax2):
        ax.axvline(12, color="orange", linestyle="--", alpha=0.6)
        ax.axvline(24, color="orange", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"acf_pacf_{zip_code}.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {zip_code}")
print(f"\nAll plots in {OUTPUT_DIR}/")

Saved 02126


Saved 02116


Saved 02150


Saved 02139


Saved 01840


Saved 02481

All plots in C:\Users\tanma\Desktop\DS4420\housingAffordability\outputs\acf_pacf/
